# YouTube Trust & Safety: Coordinated Inauthentic Behavior Detection

## Phase 2 — Real-World Detection 

### Section 1: Setup & Data Loading

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Real YouTube video statistics with 45.5 million rows tracking 270,133 unique videos over time 
yt_df = pd.read_csv("/Users/shamilikolli/Downloads/videoStatististicsALL_from21_anonymized.csv", sep=";")

In [4]:
# Convert timestamp to datetime for time-series analysis
yt_df["created_at"] = pd.to_datetime(yt_df["created_at"])

In [5]:
# Quick data info check
print(f"Full dataset rows : {len(yt_df):,}")
print(f"Unique videos : {yt_df['video_id'].nunique():,}")
print(f"Date range : {yt_df['created_at'].min()} -> {yt_df['created_at'].max()}")
print(f"Avg rows per video : {len(yt_df)/yt_df['video_id'].nunique():.1f}")
print(f"Memory usage: {yt_df.memory_usage(deep=True).sum()/1024/1024:.1f} MB")

Full dataset rows : 45,593,755
Unique videos : 270,133
Date range : 2021-01-01 00:00:28 -> 2022-05-10 11:01:11
Avg rows per video : 168.8
Memory usage: 2435.0 MB


### Sampling the data

In [ ]:
### The full dataset contains 45.5M rows across 270K videos — too large to process on a local machine. 
### For this analysis, I sampled 10,000 video IDs uniformly at random from the full set of 270K videos, 
### which yielded 1.68M rows - representing the full time-series history of those videos.

In [7]:
sampled_video_ids = yt_df["video_id"].drop_duplicates().sample(10000, random_state=42)
yt_sample = yt_df[yt_df["video_id"].isin(sampled_video_ids)].copy()

print(f"Sampled videos   : {yt_sample['video_id'].nunique():,}")
print(f"Total rows       : {len(yt_sample):,}")
print(f"Memory usage     : {yt_sample.memory_usage(deep=True).sum()/1024/1024:.1f} MB")
print(f"Date range       : {yt_sample['created_at'].min()} → {yt_sample['created_at'].max()}")

Sampled videos   : 10,000
Total rows       : 1,685,568
Memory usage     : 102.9 MB
Date range       : 2021-01-01 00:00:28 → 2022-05-10 11:01:11


## Feature Engineering

In [ ]:
### Raw numbers (likes, views) are not providing meaning alone. 
### A video with 1,000 likes could be normal or suspicious depending on its size. 
### I engineer ratios and growth features that measure relative strangeness, not absolute size.


## 1. Engagement Ratios (like_view_ratio, comment_view_ratio)
# Measures likes/comments relative to views. 50 likes on 100 views (50%) is suspicious. 50 likes on 5M views (0.001%) is normal.
# Raw '50 likes' tells us nothing. The ratio tells us everything.

## 2. Spike Detection (max_like_spike, max_view_spike)
# Real engagement grows gradually. Bots may dump thousands of likes in one hour then stop. We aim to detect the single largest
# hour-over-hour jump for each video.

##3. Spike Ratio (like_spike_ratio, view_spike_ratio)
# Spike divided by average growth. A small creator jumping from 2 likes/hour to 200 likes/hour (100x spike) is obviously botted.
# Mr X jumping from 5,000 to 6,000 (1.2x) could be just a busy hour. This makes one rule work for all video sizes.#

In [ ]:
yt_sample["like_view_ratio"] = yt_sample["likes"] / (yt_sample["views"] + 1) # Prevent division by zero with +1
yt_sample["comment_view_ratio"] = yt_sample["comments"] / (yt_sample["views"] + 1)
yt_sample["dislike_view_ratio"] = yt_sample["dislikes"] / (yt_sample["views"] + 1)


In [9]:
print("Sample ratios:")
print(yt_sample[["video_id","views","likes","like_view_ratio"]].head(5))

Sample ratios:
      video_id  views  likes  like_view_ratio
2040    100012     81    5.0         0.060976
2041    100012    152   18.0         0.117647
2042    100012    184   22.0         0.118919
2043    100012    598   23.0         0.038397
2044    100012    897   24.0         0.026726


In [19]:
yt_sample.to_csv("data/processed/yt_sample.csv", index=False)

In [ ]:
# TIME-SERIES FEATURE ENGINEERING
# We aggregate each video's full history into a single feature vector.

In [16]:
def engineer_video_features(df):
    features = []
    
    for video_id, group in df.groupby("video_id"):
        group = group.sort_values("created_at").reset_index(drop=True)  # Reset index for clean positions
        
        if len(group) < 2:
            continue
        
        # Calculate growth with reset index
        view_growth = group["views"].diff().fillna(0)
        like_growth = group["likes"].diff().fillna(0)
        
        # Find peak hour using position (iloc), not label (loc)
        peak_idx = view_growth.idxmax()
        peak_hour = group.loc[peak_idx, "created_at"].hour if peak_idx in group.index else group.iloc[0]["created_at"].hour
        
        # Maximum single-hour spikes
        max_like_spike = like_growth.max()
        max_view_spike = view_growth.max()
        
        # Average growth
        avg_like_growth = like_growth.mean()
        avg_view_growth = view_growth.mean()
        
        # Spike ratios
        like_spike_ratio = max_like_spike / (avg_like_growth + 1)
        view_spike_ratio = max_view_spike / (avg_view_growth + 1)
        
        features.append({
            "video_id": video_id,
            "total_views": group["views"].max(),
            "total_likes": group["likes"].max(),
            "total_comments": group["comments"].max(),
            "total_dislikes": group["dislikes"].max(),
            "like_view_ratio": group["likes"].max() / (group["views"].max() + 1),
            "comment_view_ratio": group["comments"].max() / (group["views"].max() + 1),
            "dislike_view_ratio": group["dislikes"].max() / (group["views"].max() + 1),
            "max_like_spike": max_like_spike,
            "max_view_spike": max_view_spike,
            "like_spike_ratio": like_spike_ratio,
            "view_spike_ratio": view_spike_ratio,
            "peak_hour": peak_hour,
            "hours_tracked": len(group),
        })
    
    return pd.DataFrame(features)

video_features = engineer_video_features(yt_sample)
print(video_features.head())

   video_id  total_views  total_likes  total_comments  total_dislikes  \
0    100012         3924         52.0            10.0             3.0   
1    100087        22384        147.0           257.0            98.0   
2    100088         5198         89.0            46.0            10.0   
3    100106         6094         93.0             6.0             2.0   
4    100119        34569        304.0           176.0            69.0   

   like_view_ratio  comment_view_ratio  dislike_view_ratio  max_like_spike  \
0         0.013248            0.002548            0.000764            13.0   
1         0.006567            0.011481            0.004378            13.0   
2         0.017119            0.008848            0.001923            13.0   
3         0.015258            0.000984            0.000328            18.0   
4         0.008794            0.005091            0.001996            58.0   

   max_view_spike  like_spike_ratio  view_spike_ratio  peak_hour  \
0           414.0       

### save processed features

In [18]:
video_features.to_csv("data/processed/video_features_engineered.csv", index=False)

In [17]:
video_features.shape

(9999, 14)